# EDEF Tree Regression Quickstart

For tree models, EDEF uses exact TreeIG path traces.

For squared-error regression, each split crossing contributes the finite reduction in squared-error loss induced by the prediction jump at that crossing.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

import edef


In [ ]:
rng = np.random.default_rng(123)

X = rng.normal(size=(500, 4))

y = (
    1.0 * X[:, 0]
    - 0.8 * X[:, 1] ** 2
    + 0.5 * X[:, 2]
    + rng.normal(scale=0.1, size=500)
)

feature_names = [
    "x1_linear",
    "x2_squared",
    "x3_linear",
    "x4_noise",
]


In [ ]:
model = RandomForestRegressor(
    n_estimators=20,
    max_depth=4,
    random_state=0,
    n_jobs=1,
)

model.fit(X, y)


In [ ]:
explainer = edef.TreeExplainer(
    model,
    baseline=X.mean(axis=0),
    loss="squared_error",
    feature_names=feature_names,
)

result = explainer(X, y)


In [ ]:
print(result)


In [ ]:
result.to_frame()


In [ ]:
ax = result.plot()
ax.set_title("Tree EDEF contributions to realized squared-error fit")
plt.show()


In [ ]:
grouped = result.group(
    ["signal", "signal", "signal", "noise"]
)

print(grouped)


In [ ]:
ax = grouped.plot()
ax.set_title("Grouped tree EDEF contributions")
plt.show()


TreeExplainer relies on TreeIG to compute exact split-crossing prediction jumps.

EDEF then converts each prediction jump into a finite loss-improvement contribution.


## SHAP Plotting Compatibility


In [ ]:
shap_exp = result.to_shap_explanation(data=X)


In [ ]:
import shap

shap.plots.beeswarm(shap_exp)
